# BERT title/subtitle/intro + max-history similarity + soft freshness

Reuse your existing frozen title/subtitle/intro DistilBERT vectors. For each candidate, take its maximum cosine similarity to the user's latest 10 available past-click events. Compare that baseline with a soft publication-age penalty and a freshness-only control.

`score = max_history_similarity - lambda * log1p(age_hours)`, where age is measured at the recommendation request, not the current clock time. Lambda values are 0, 0.01, 0.03 and 0.1. Freshness-only uses `-log1p(age_hours)`. Eligible pools remain unchanged; no abrupt age cutoff is introduced.

Evaluation and weight selection use **Recall@100 and nDCG@100**. Exports retain **200** candidates in native Parquet, highest-ranked first, for the group reranker. This is a scoring-rule experiment: no new embeddings, fine-tuning or model training are performed.


In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import importlib.util
import json
import subprocess
import sys

try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False
EXPERIMENT_ID = '3.6_bert_intro_freshness'
if IN_COLAB:
    from google.colab import drive, files
    drive.mount('/content/drive')
    CODE_DIR = Path('/content/Experiments') / EXPERIMENT_ID
    CODE_DIR.mkdir(parents=True, exist_ok=True)
    print('Upload BOTH catalogue_pipeline.py and freshness_pipeline.py from this experiment folder.')
    uploaded = files.upload()
    for module in ['catalogue_pipeline', 'freshness_pipeline']:
        matches = [name for name in uploaded if Path(name).stem.startswith(module) and Path(name).suffix == '.py']
        if len(matches) != 1: raise ValueError(f'Upload exactly one {module}.py')
        (CODE_DIR / f'{module}.py').write_bytes(uploaded[matches[0]])
else:
    choices = [Path.cwd(), Path.cwd() / 'Experiments' / EXPERIMENT_ID]
    CODE_DIR = next((p for p in choices if (p / 'freshness_pipeline.py').is_file()), None)
    if CODE_DIR is None: raise FileNotFoundError('Open from the repository root or this experiment folder.')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy>=1.26', 'pandas>=2.2',
                'pyarrow>=15', 'scikit-learn>=1.4'], check=True)
import pandas as pd

def load_module(name):
    spec = importlib.util.spec_from_file_location(name, CODE_DIR / f'{name}.py')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module
cp = load_module('catalogue_pipeline')
fp = load_module('freshness_pipeline')
if fp.VERSION != 'bert_intro_max_soft_freshness_at100_v1':
    raise RuntimeError('Upload freshness_pipeline.py from Experiments/3.6_bert_intro_freshness.')
print('Loaded:', cp.PIPELINE_VERSION, fp.VERSION)


Mounted at /content/drive
Upload BOTH catalogue_pipeline.py and freshness_pipeline.py from this experiment folder.


Saving catalogue_pipeline.py to catalogue_pipeline.py
Saving freshness_pipeline.py to freshness_pipeline.py
Loaded: similarity_individual_history_v6 bert_intro_max_soft_freshness_at100_v1


## Point to your existing BERT intro embeddings

Set DRIVE_ROOT to the same smaller dataset used by your selected BERT intro/max-history run. BERT_CACHE defaults to the cache path shown in that notebook's saved outputs. If your actual cache lives elsewhere, change that one path. The cell checks the encoder, text variant, completion flag and saved dimension; it never calls an embedding model or API.

Keep seed 42, latest-10 max-history aggregation and the same 1,000 validation request IDs. The metadata records how many body characters were included (normally 1,000). If reusing vectors from a larger lookup, article IDs/text must still match; the candidate catalogue continues to come from the current shared article files. Audit candidate and history-vector coverage before running.

Use a fresh run ID for this experiment. After audit, set RUN_PHASE='run' in the separate phase cell, preserving WORK/RUN_ID. Do not enable additional test-based tuning.


In [2]:
DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder')

# Existing FROZEN BERT intro cache. No new embeddings are generated here.
BERT_CACHE = (
    DRIVE_ROOT / 'Experiments' / '3.2_pretrained_similarity_bert_title_subtitle_intro'
    / 'embeddings' / 'distiluse-base-multilingual-cased-v2_title_subtitle_intro_1000chars'
)
metadata_path = BERT_CACHE / 'embedding_config.json'
if not metadata_path.is_file():
    raise FileNotFoundError('Set BERT_CACHE to your existing title/subtitle/intro cache folder.')
metadata = json.loads(metadata_path.read_text())
if metadata.get('text_variant') != 'title_subtitle_intro':
    raise ValueError('This experiment requires the title_subtitle_intro embedding cache.')
if metadata.get('model') != 'sentence-transformers/distiluse-base-multilingual-cased-v2':
    raise ValueError('Cache encoder differs from the selected DistilBERT experiment. Check its provenance.')
if not metadata.get('complete'):
    raise ValueError('Embedding cache is incomplete. Finish its generation before this experiment.')
embedding_paths = {
    f'embeddings_{part}': str(BERT_CACHE / f'{part}_text_embeddings.parquet')
    for part in ('train_val', 'test')
}
for path in embedding_paths.values():
    if not Path(path).is_file(): raise FileNotFoundError(path)

RUN_ID = datetime.now(ZoneInfo('Asia/Singapore')).strftime('%Y%m%d_%H%M%S') + '_bert_intro_max_freshness_seed42'
RUN_PHASE = 'audit'
WORK = CODE_DIR / 'outputs' / RUN_ID
CONFIG = {
    'experiment_id': EXPERIMENT_ID, 'run_id': RUN_ID, 'owner': '',
    'drive_root': str(DRIVE_ROOT), 'seed': 42, 'scope': 'development',
    'input_version': None,
    'embedding_source': {
        'provider': 'Sentence Transformers', 'model': metadata['model'],
        'revision': metadata['model_revision'], 'text_variant': metadata['text_variant'],
        'intro_characters': metadata['intro_characters'], 'pooling': metadata['pooling'],
        'metadata_path': str(metadata_path), 'fine_tuned': False,
    },
    'embedding_column': 'text_embedding', 'embedding_dimension': metadata['dimension'],
    'history_limit': 10, 'history_aggregation': 'max_history',
    'history_top_k': 3,  # Ignored for max_history: each candidate uses its ONE highest cosine.
    'validation_requests': 1000,
    'export_requests': {'train': 25, 'validation': 1000, 'test': 25},
    'freshness_weights': [0.0, 0.01, 0.03, 0.1],
    'catalogue_batch_size': 4096,
    'similarity_device': 'auto', 'similarity_min_gpu_batch': 256,
    'hash_inputs': False, 'max_index_gb': 20,
    'max_export_pairs': 150_000_000, 'max_export_gb': 10,
    'persistent_root': str(DRIVE_ROOT / 'Experiments'),
    'paths': embedding_paths,
}
print('Drive root exists:', DRIVE_ROOT.exists())
print('Reusing frozen BERT embeddings:', BERT_CACHE)
print('Encoder revision:', metadata['model_revision'])
print('Intro characters:', metadata['intro_characters'])
print('Local run:', WORK)


Drive root exists: True
Reusing frozen BERT embeddings: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/3.2_pretrained_similarity_bert_title_subtitle_intro/embeddings/distiluse-base-multilingual-cased-v2_title_subtitle_intro_1000chars
Encoder revision: bfe45d0732ca50787611c0fe107ba278c7f3f889
Intro characters: 1000
Local run: /content/Experiments/3.6_bert_intro_freshness/outputs/20261105_212000_bert_intro_max_freshness_seed42


In [3]:
PREPARED = cp.prepare(CONFIG, WORK)
audit = PREPARED['audit']
print('Catalogue articles:', audit['catalogue_articles'])
print('Catalogue articles without vectors:', audit['catalogue_articles_without_vectors'])
print('Embedding inputs:', {k: v for k, v in PREPARED['inputs'].items() if k.startswith('embeddings_')})
print('Similarity compute:', audit['compute'])
print('Validation requests:', len(PREPARED['validation_requests']))
display(pd.DataFrame(audit['planned_exports']).T)
display(pd.DataFrame(audit['full_export_costs']).T)
cp.check_cost(PREPARED)


Similarity-only compute: {'requested_device': 'auto', 'similarity_device': 'cuda', 'classifier_device': 'not_applicable', 'loading_and_exports_device': 'cpu', 'minimum_gpu_batch': 256, 'reason': 'Shared CUDA article table ready', 'article_table_bytes': 42436608, 'torch_version': '2.11.0+cu130', 'cuda_version': '13.0', 'gpu_name': 'NVIDIA L4'}
Catalogue articles: 20738
Catalogue articles without vectors: 18
Embedding inputs: {'embeddings_train_val': PosixPath('/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/3.2_pretrained_similarity_bert_title_subtitle_intro/embeddings/distiluse-base-multilingual-cased-v2_title_subtitle_intro_1000chars/train_val_text_embeddings.parquet'), 'embeddings_test': PosixPath('/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/3.2_pretrained_similarity_bert_title_subtitle_intro/embeddings/distiluse-base-multilingual-cased-v2_title_subtitle_intro_1000chars/test_text_embeddings.parquet')}
Similarity compute:

,requests,eligible_pairs,empty_pools,shortlisted_articles,estimated_csv_bytes,csv_estimate_note,catalogue_articles,missing_publication_excluded,runtime_seconds,runtime_note
train,25,463808,0,5000,282500,Top-200 IDs and aligned similarity scores plus...,20738,0,None,Measure a bounded sample; no reliable timing e...
validation,1000,19461013,0,200000,11300000,Top-200 IDs and aligned similarity scores plus...,20738,0,None,Measure a bounded sample; no reliable timing e...
test,25,486145,0,5000,282500,Top-200 IDs and aligned similarity scores plus...,20738,0,None,Measure a bounded sample; no reliable timing e...


,requests,eligible_pairs,empty_pools,shortlisted_articles,estimated_csv_bytes,csv_estimate_note,catalogue_articles,missing_publication_excluded,runtime_seconds,runtime_note
train,232887,4319178324,0,46577400,2631623100,Top-200 IDs and aligned similarity scores plus...,20738,0,None,Measure a bounded sample; no reliable timing e...
validation,29110,566218581,0,5822000,328943000,Top-200 IDs and aligned similarity scores plus...,20738,0,None,Measure a bounded sample; no reliable timing e...
test,29142,566598922,0,5828400,329304600,Top-200 IDs and aligned similarity scores plus...,20738,0,None,Measure a bounded sample; no reliable timing e...


{'pairs': 20410966,
 'estimated_csv_bytes': 11865000,
 'local_free_bytes': 252280573952}

## Compare formulas and select on validation

After the audit, run the next cell to switch phase without changing RUN_ID. Similarities are calculated once per candidate batch and reused for all weights and the freshness-only control. Every formula still ranks the complete eligible pool. Selection maximises mean known-click Recall@100, then observed-click nDCG@100. Exact ties preserve the earlier option: unmodified similarity first, increasing penalties next, freshness-only last.

Known clicks excluded by eligibility remain in recall denominators and nDCG ideals. Unshown article outcomes stay unknown. Missing similarity/profile vectors retain the original zero fallback, after which the same age penalty is applied. Freshness-only can win; that is a substantive result rather than evidence that embeddings improved. Test labels do not select the weight.


In [4]:
RUN_PHASE = 'run'  # Run after reviewing the audit.


In [5]:
GRID = None
if RUN_PHASE == 'run':
    GRID = fp.evaluate_grid(PREPARED)
    comparison = pd.DataFrame([
        {'variant': row['variant']['name'], 'lambda': row['variant']['weight'],
         'recall@100': row['metrics']['known_click_recall@100'],
         'ndcg@100': row['metrics']['observed_click_ndcg@100'],
         'validation_requests': row['metrics']['evaluated_requests']}
        for row in GRID['results']
    ])
    display(comparison)
    print('Selected rule:', GRID['selected']['variant'])
    print('Candidate pairs scored once:', GRID['candidate_pairs_scored_once'])
    print('Grid seconds:', round(GRID['seconds'], 2))


Freshness validation: 100 requests; 4.2s
Freshness validation: 200 requests; 8.3s
Freshness validation: 300 requests; 12.4s
Freshness validation: 400 requests; 16.4s
Freshness validation: 500 requests; 20.4s
Freshness validation: 600 requests; 24.5s
Freshness validation: 700 requests; 28.6s
Freshness validation: 800 requests; 32.7s
Freshness validation: 900 requests; 36.9s
Freshness validation: 1,000 requests; 41.2s


,variant,lambda,recall@100,ndcg@100,validation_requests
0,similarity_only,0.00,0.0150,0.002737,1000
1,similarity_lambda_0p01,0.01,0.0420,0.007830,1000
2,similarity_lambda_0p03,0.03,0.1530,0.029725,1000
3,similarity_lambda_0p1,0.10,0.7445,0.163928,1000
4,freshness_only,NaN,0.9090,0.249783,1000


Selected rule: {'name': 'freshness_only', 'kind': 'freshness_only', 'weight': None}
Candidate pairs scored once: 19461013
Grid seconds: 41.21


## Export the validation-selected formula as native Parquet

Only the selected formula is exported. `rankings/<split>/submission_top_200.parquet` has exactly `behaviour_id` and native integer-list `article_ids`, one request per row, highest-ranked article first. No submission CSV is generated.

`submission_top_200_scores.parquet` stores the same ordered IDs plus `scores` (FINAL ranking values), raw `similarity_scores`, request-relative `article_age_hours`, eligible-pool counts and boundary-tie flags. Final scores can be below -1 and are not probabilities. Raw similarities need not be descending after freshness changes ranking.

Each saved shortlist and score companion is independently recomputed against its full eligible pool. Exported-validation metrics then read the saved Parquet lists without an additional catalogue-scoring pass. Test exports ignore test click labels. The separate shared evaluator can assess genuine test labels after the formula/weight are fixed. This notebook does not perform Stage 2 business reranking or report AUC.

Defaults export 25 train, 1,000 validation and 25 test requests. To change scope, use a fresh prepared run and inspect the pair/disk budget. Exporting all validation and test requests on the smaller catalogue can exceed one billion candidate pairs. Sampling requests never reduces the eligible catalogue searched for each selected request.


In [6]:
EXPORT = None
if RUN_PHASE == 'run' and GRID is not None:
    EXPORT = fp.export_selected(PREPARED, GRID)
    display(pd.json_normalize(EXPORT['exported_validation']))
    print('Export checks:', EXPORT['checks'])
    print('Shortlists:', WORK / 'rankings')
    print('Manifest:', WORK / 'run_config.json')


,known_click_recall@100,observed_click_ndcg@100,evaluated_requests,requests_scored,requests_without_known_clicks,known_clicks,known_clicks_excluded_by_eligibility,known_click_eligibility_coverage,eligible_only_recall@100,empty_eligible_pools,requests_with_ties_crossing_100,requests_without_usable_user_profile,outcomes,source,known_click_exclusion_reasons.missing_article,known_click_exclusion_reasons.missing_publication,known_click_exclusion_reasons.future_publication
0,0.909,0.249783,1000,1000,0,1006,0,1.0,0.909,0,4,0,Unshown outcomes unknown; original known-click...,saved and independently verified native Parque...,0,0,0


Export checks: {'train': {'requests_verified': 25, 'shortlisted_articles_verified': 5000, 'catalogue_pairs_rescored_for_verification': 463808, 'full_score_rows_exported': 0}, 'validation': {'requests_verified': 1000, 'shortlisted_articles_verified': 200000, 'catalogue_pairs_rescored_for_verification': 19461013, 'full_score_rows_exported': 0}, 'test': {'requests_verified': 25, 'shortlisted_articles_verified': 5000, 'catalogue_pairs_rescored_for_verification': 486145, 'full_score_rows_exported': 0}}
Shortlists: /content/Experiments/3.6_bert_intro_freshness/outputs/20261105_212000_bert_intro_max_freshness_seed42/rankings
Manifest: /content/Experiments/3.6_bert_intro_freshness/outputs/20261105_212000_bert_intro_max_freshness_seed42/run_config.json


## Copy the completed bundle to Drive

This copies comparison results, per-request validation metrics, selected shortlists, the run manifest, helper code and package versions. Large generated SQLite indexes are omitted. Existing destinations are preserved; use a fresh copy suffix after a failed partial copy rather than silently overwriting another run. Copy completeness is checked by file sizes; this is not a saved-neural-model reload test because the rule has no learned weights.


In [7]:
if EXPORT is not None:
    SAVED = fp.persist(PREPARED)
    print('Saved run:', SAVED)


Saved run: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/3.6_bert_intro_freshness/20261105_212000_bert_intro_max_freshness_seed42


## Interpret the result

Compare max-history similarity alone, freshness alone and the selected blend on identical requests. A blend that improves over both controls supports complementary content and age signals. A freshness-only win is also informative; it does not demonstrate better embeddings. Report the weight, history rule, catalogue size, request IDs, coverage and cutoff.

The no-freshness row is this notebook's BERT intro/max-history baseline at @100. Do not compare it directly with an earlier @200 number. When comparing against your Gemini top-3 freshness experiment, note that both encoder AND history aggregation differ; that is a comparison of selected retriever configurations, not an encoder-only ablation. Keep the catalogue/requests/eligibility and metric cutoff the same.

Use validation for selection. Final held-out test evaluation is separate. The evaluator/reranker must use the `scores` column in the score companion, or preserve the canonical exported order, rather than reordering by raw similarities.
